# Location Selection with E-NAUTILUS: Part 2
_Running of E-NAUTILUS for decision making, and presentation of results_


In [5]:
import numpy as np
import pandas as pd
import polars as pl
import pickle
import folium
# These are to just suppress warnings in the outputs of the example
import warnings

warnings.filterwarnings("ignore")

## Load results from previous session

In [6]:
file_name = "data/pf_16.pkl"

output = open(file_name, 'rb')
prev_session = pickle.load(output)

raw_ref_pf = prev_session["pf"]
prob = prev_session["prob"]
events = prev_session["events"]
cities = prev_session["cities"]
event2city = prev_session["event2city"]


## Load reference front and problem 

In [8]:
output_flat = np.array(raw_ref_pf).flatten()

def process_lists(dict2conv):
    return {key: np.array(dict2conv[key]).flatten().tolist() for key in dict2conv.keys()}

# TODO include constraints in here too
output_dict = [
    output.optimal_objectives | 
    process_lists(output.optimal_variables) 
    for output in output_flat]

nd_df = pl.DataFrame(output_dict)

nd_df = nd_df.unique(subset=("f_1", "f_2", "f_3", "f_4"))

nd_df = nd_df.with_columns([
    (-pl.col("f_1")).alias("f_1_min"),
    (pl.col("f_2")).alias("f_2_min"),
    (pl.col("f_3")).alias("f_3_min"),
    (-pl.col("f_4")).alias("f_4_min")
])

nadir_point = {
  "f_1": float(nd_df["f_1"].min()),
  "f_2": float(nd_df["f_2"].max()),
  "f_3": float(nd_df["f_3"].max()),
  "f_4": float(nd_df["f_4"].min())
}

display(nd_df)

print(f"Nadir point: {nadir_point}")
print(f"Nadir point (problem): {prob.get_nadir_point()}")
print(f"Idedal point (problem): {prob.get_ideal_point()}")


reachable_indices = list(range(len(nd_df)))  # everything reachable from nadir


f_1,f_2,f_3,f_4,ev,cover,_alpha,f_1_min,f_2_min,f_3_min,f_4_min
f64,f64,f64,f64,list[f64],list[f64],list[f64],f64,f64,f64,f64
441.0,10.0,1600.065,0.393777,"[1.0, 1.0, … 1.0]","[1.0, 0.0, … 0.0]",[1.899771],-441.0,10.0,1600.065,-0.393777
432.0,4.0,1054.855,0.412698,"[1.0, 0.0, … 0.0]","[1.0, 1.0, … 0.0]",[0.660679],-432.0,4.0,1054.855,-0.412698
206.0,5.0,804.15,0.439283,"[1.0, 0.0, … 0.0]","[1.0, 1.0, … 0.0]",[0.560716],-206.0,5.0,804.15,-0.439283
437.0,6.0,1247.235,0.425879,"[1.0, 1.0, … 0.0]","[1.0, 1.0, … 0.0]",[0.69742],-437.0,6.0,1247.235,-0.425879
0.0,0.0,0.0,0.0,"[0.0, 0.0, … 0.0]","[0.0, 0.0, … 0.0]",[-9.9600e-9],-0.0,0.0,0.0,-0.0
…,…,…,…,…,…,…,…,…,…,…
332.0,5.0,884.875,0.439283,"[1.0, 0.0, … 0.0]","[1.0, 1.0, … 0.0]",[0.560716],-332.0,5.0,884.875,-0.439283
120.0,6.0,713.325,0.443013,"[1.0, 0.0, … 1.0]","[1.0, 1.0, … 0.0]",[838.355363],-120.0,6.0,713.325,-0.443013
355.0,0.0,445.7,0.247049,"[0.0, 0.0, … 0.0]","[0.0, 0.0, … 0.0]",[0.278551],-355.0,0.0,445.7,-0.247049


Nadir point: {'f_1': 0.0, 'f_2': 10.0, 'f_3': 1600.065, 'f_4': 0.0}
Nadir point (problem): {'f_1': 0, 'f_2': 10, 'f_3': 1600.065, 'f_4': 0}
Idedal point (problem): {'f_1': 441, 'f_2': 0, 'f_3': 0, 'f_4': 1.0}


## Helper functions

In [20]:
def no_nan(val):
    if pd.isna(val):
        return ""
    else:
        return str(val)
    
# Function to determine marker size based on population
def get_marker_size(population):
    return max(5, population / 1000)  # Adjust the divisor to scale marker size

def create_color_dict(cities, ev_cities, cc): 
    marker_color = {}
    for city in cities.loc[:,"city"]: 
        if city in ev_cities: 
            marker_color[city] = "orange"
        elif city in cc: 
            marker_color[city] = "yellow"
        else: 
            marker_color[city] = "grey"

    return marker_color

def select_point(results, sol_id): 

    return {
        "f_1": int(results.loc[sol_id, "Total patients served"]),
        "f_2": int(results.loc[sol_id, "Number of overstaffed events"]),
        "f_3": float(results.loc[sol_id, "Total costs ($)"]),
        "f_4": float(results.loc[sol_id, "Population with access (%)"]/100.0)
        }


def clean_results(raw_results, intermediate_point=True): 
    # Transform objectives
    if intermediate_point: 
        results = pd.DataFrame(raw_results.intermediate_points)
    else:
        results = pd.DataFrame(raw_results.optimal_objectives)

    results = results.rename(columns={
                    "f_1": "Total patients served", 
                    "f_2": "Number of overstaffed events", 
                    "f_3": "Total costs ($)", 
                    "f_4": "Population with access (%)"})
    results[["Total patients served", "Number of overstaffed events"]] =  results[["Total patients served", "Number of overstaffed events"]].astype(int)
    results[["Population with access (%)"]] = (results[["Population with access (%)"]]*100.0).round(2)
    results[["Total costs ($)"]] = (results[["Total costs ($)"]]).round(2)

    results.index.name = "Solution ID"

    return results

## Run eNAUTILUS 
### Round 1
We're going to generate some solutions. They will be poor at first, but you and the computer will slowly find the best solution that fulfills your goals and preferences. 


In [30]:
from desdeo.mcdm.enautilus import enautilus_step
from desdeo.mcdm.enautilus import enautilus_get_representative_solutions

current_iter = 0
selected_point = nadir_point
total_iterations = 3
display(f"Starting with point {selected_point}")

prob.get_ideal_point()


raw_results = enautilus_step(
    problem=prob,
    non_dominated_points=nd_df,
    current_iteration=current_iter,
    iterations_left=total_iterations - current_iter,
    selected_point=selected_point,
    reachable_point_indices=reachable_indices,
    total_number_of_iterations=total_iterations,
    number_of_intermediate_points=3,
)

print(f"number of iterations left: {total_iterations - current_iter}")

results = clean_results(raw_results)
display("Which solution to do you prefer?")
display(results)

"Starting with point {'f_1': 0.0, 'f_2': 10.0, 'f_3': 1600.065, 'f_4': 0.0}"

number of iterations left: 3


'Which solution to do you prefer?'

,Total patients served,Number of overstaffed events,Total costs ($),Population with access (%)
Solution ID,,,,
0,110,8,1361.67,14.64
1,147,10,1600.06,13.13
2,0,6,1066.71,0.00


### Round 2 

In [31]:
chosen_solution = 1

current_iter += 1
selected_point = select_point(results, chosen_solution)

print(selected_point)

raw_results = enautilus_step(
    problem=prob,
    non_dominated_points=nd_df,
    current_iteration=current_iter,
    iterations_left=total_iterations - current_iter,
    selected_point=selected_point,
    reachable_point_indices=reachable_indices,
    total_number_of_iterations=total_iterations,
    number_of_intermediate_points=3,
)

print(f"number of iterations left: {total_iterations - current_iter}")
display(raw_results)
results = clean_results(raw_results)
display("Which solution to do you find most preferable?")
display(results)
display("Results:")



{'f_1': 147, 'f_2': 10, 'f_3': 1600.06, 'f_4': 0.1313}
number of iterations left: 2


ENautilusResult(current_iteration=2, iterations_left=1, intermediate_points=[{'f_1': 239.5, 'f_2': 7.5, 'f_3': 1242.4675, 'f_4': 0.28529168249121895}, {'f_1': 294.0, 'f_2': 10.0, 'f_3': 1600.0625, 'f_4': 0.26253845862655295}, {'f_1': 73.5, 'f_2': 5.0, 'f_3': 800.03, 'f_4': 0.06565}], reachable_best_bounds=[{'f_1': 432.0, 'f_2': 0.0, 'f_3': 622.69, 'f_4': 0.4430129947499721}, {'f_1': 437.0, 'f_2': 0.0, 'f_3': 622.69, 'f_4': 0.4430129947499721}, {'f_1': 400.0, 'f_2': 0.0, 'f_3': 445.7, 'f_4': 0.43928336498243786}], reachable_worst_bounds=[{'f_1': 239.5, 'f_2': 7.5, 'f_3': 1242.4675, 'f_4': 0.28529168249121895}, {'f_1': 294.0, 'f_2': 10.0, 'f_3': 1600.0625, 'f_4': 0.26253845862655295}, {'f_1': 73.5, 'f_2': 5.0, 'f_3': 800.03, 'f_4': 0.06565}], closeness_measures=[54.58381724236615, 66.66666667323062, 50.21071241325801], reachable_point_indices=[[1, 2, 5, 6, 7, 8, 9, 10, 13, 14], [1, 2, 3, 5, 6, 7, 8, 9, 10, 13, 14], [2, 7, 9, 10, 12, 13]])

'Which solution to do you find most preferable?'

,Total patients served,Number of overstaffed events,Total costs ($),Population with access (%)
Solution ID,,,,
0,239,7,1242.47,28.53
1,294,10,1600.06,26.25
2,73,5,800.03,6.56


'Results:'

### Round 3

In [32]:

chosen_solution = 0

current_iter += 1
selected_point = select_point(results, chosen_solution)

print(selected_point)

raw_results = enautilus_step(
    problem=prob,
    non_dominated_points=nd_df,
    current_iteration=current_iter,
    iterations_left=total_iterations - current_iter,
    selected_point=selected_point,
    reachable_point_indices=reachable_indices,
    total_number_of_iterations=total_iterations,
    number_of_intermediate_points=3,
)

print(f"number of iterations left: {total_iterations - current_iter}")

results = clean_results(raw_results)
display("Which solution to do you find most preferable?")
display(results)




{'f_1': 239, 'f_2': 7, 'f_3': 1242.47, 'f_4': 0.2853}
number of iterations left: 1


'Which solution to do you find most preferable?'

,Total patients served,Number of overstaffed events,Total costs ($),Population with access (%)
Solution ID,,,,
0,332,5,884.88,43.93
1,441,10,1600.06,39.38
2,0,0,0.00,0.00


## Display final result

In [33]:
final_chosen_solution = 0

solutions = enautilus_get_representative_solutions(prob, raw_results, nd_df) 
solution = solutions[final_chosen_solution]
results = clean_results(solution, intermediate_point=False)

display(results)
 


,Total patients served,Number of overstaffed events,Total costs ($),Population with access (%)
Solution ID,,,,
0,332,5,884.88,43.93


### Result postprocessing

In [34]:

raw_events = solution.optimal_variables['ev'][0].to_list()
raw_events = [[bool(e) for e in raw_events]]

raw_coverage = solution.optimal_variables['cover'][0].to_list()
raw_coverage = [[bool(c) for c in raw_coverage]]

events_visited = []
for evb in raw_events: 
    events_visited.append("\n".join(events.loc[evb, "event_id"].values))

cities_covered = [] 
for cc in raw_coverage: 
    cities_covered.append("\n".join(cities.loc[cc,"city"].values))

results["Events Visited"] = events_visited
results["Cities covered"] = cities_covered

results

,Total patients served,Number of overstaffed events,Total costs ($),Population with access (%),Events Visited,Cities covered
Solution ID,,,,,,
0,332,5,884.88,43.93,ada-public-library\nbluffton-bluffton-public-l...,Ada\nAlger\nBluffton\nCairo\nColumbus Grove\nC...


### Map preprocessing

In [35]:
events_in_cities = events.loc[raw_events[0],:].groupby("city").agg({"event_pretty": lambda e : '<br>'.join(e)})
events_in_cities = events_in_cities.to_dict()['event_pretty']
events_in_cities

# cc cities covered
cc = set(cities_covered[0].split('\n'))

# event_cities 
ev_cities = list(events.loc[raw_events[0], "city"])
marker_colors = create_color_dict(cities, ev_cities, cc)


## Event coverage dictionary 
event2city_mat = event2city[raw_events[0]].astype(bool)

event2city_dict = {}
for (c,city) in enumerate(ev_cities): 
    event2city_dict[city] = set(cities.loc[event2city_mat[c,],"city"]) - {city}

adjacent_events = {}
# Record what events are near cities
for event_city in event2city_dict.keys(): 
    adj_cities = event2city_dict[event_city]
    from_name = cities.loc[cities.loc[:,"city"] == event_city,["city"]].values.tolist()[0][0]

    for adj_city in adj_cities: 
        to_name = cities.loc[cities.loc[:,"city"] == adj_city ,["city"]].values.tolist()[0][0]

        if to_name not in adjacent_events.keys(): 
            adjacent_events[to_name] = {from_name}
        else: 
            adjacent_events[to_name] = adjacent_events[to_name].union({from_name})



## Render map

In [36]:
# Create a base map
m = folium.Map(location=[cities['lat'].mean(), 
                         cities['long'].mean()], 
                         zoom_start=7) 

# Draw lines between 
for event_city in event2city_dict.keys(): 
    adj_cities = event2city_dict[event_city]
    from_loc = cities.loc[cities.loc[:,"city"] == event_city,["lat", "long"]].values.tolist()

    for adj_city in adj_cities: 
        to_loc = cities.loc[cities.loc[:,"city"] == adj_city ,["lat", "long"]].values.tolist()
        folium.PolyLine(
            locations=[to_loc[0], from_loc[0]],
            color="black"
        ).add_to(m)

# Set bounds
sw = cities.loc[:,['lat', 'long']].min().values.tolist()
ne = cities.loc[:,['lat', 'long']].max().values.tolist()
m.fit_bounds([sw,ne])

# Create tool tips 
tooltips = {}
for _, row in cities.iterrows():
    city = row['city']
    tooltips[city]=f"<b>{city}</b><br><b>Population:</b> {row['pop']}"

    if city in events_in_cities.keys():
        tooltips[city]+= "<br><b>Events:</b><br>"
        tooltips[city]+= events_in_cities[city]
    else:
        tooltips[city]+= "<br><b>No Healthwise Clinics</b>"

    if city in adjacent_events.keys(): 
        tooltips[city]+= "<br><b>Covered by events in: </b>"
        tooltips[city]+= "<br>".join(adjacent_events[city])


# Add cities to the map
for _, row in cities.iterrows():
    city = row['city']
    folium.CircleMarker(
        location=(row['lat'], row['long']),
        radius=get_marker_size(row['pop']),
        color="black",
        fill=True,
        fill_color=marker_colors[city],
        fill_opacity=0.6,
        tooltip=tooltips[city]
    ).add_to(m)

display(results)
display(m)

,Total patients served,Number of overstaffed events,Total costs ($),Population with access (%),Events Visited,Cities covered
Solution ID,,,,,,
0,332,5,884.88,43.93,ada-public-library\nbluffton-bluffton-public-l...,Ada\nAlger\nBluffton\nCairo\nColumbus Grove\nC...
